# SHAP Interpretability

This notebook explains the XGBoost churn model using SHAP (SHapley Additive
exPlanations). SHAP values fairly attribute the difference between a prediction
and the model's expected output to each input feature.

**Contents**
1. Data loading and model training
2. Build and persist the SHAP TreeExplainer
3. Global importance — bar chart and beeswarm summary
4. Dependence plots for the five most influential features
5. Local explanations for three customer archetypes: high-risk, low-risk, and borderline

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

root = Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

matplotlib.rcParams["figure.dpi"] = 120

from src.data.load import download_telco_data
from src.data.pipeline import prepare_data
from src.evaluation.interpretability import (
    build_explainer,
    clean_feature_names,
    explain_customer,
    get_shap_values,
    plot_dependence,
    plot_importance_bar,
    plot_summary,
    save_explainer,
    top_features_by_importance,
)
from src.models.gbm import train_xgboost
from src.utils.plotting import set_plot_style

set_plot_style()

FIGURES_DIR = root / "reports" / "figures"
MODELS_DIR = root / "models"
RAW_PATH = root / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
SEED = 42

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"shap {shap.__version__}  |  root: {root}")

## 1. Data and Model

In [ ]:
download_telco_data(RAW_PATH)
print("Dataset ready.")

In [ ]:
data = prepare_data(
    RAW_PATH,
    val_size=0.1,
    test_size=0.2,
    seed=SEED,
    processed_dir=root / "data" / "processed",
)

X_train, y_train = data["X_train"], data["y_train"]
X_val,   y_val   = data["X_val"],   data["y_val"]
X_test,  y_test  = data["X_test"],  data["y_test"]
raw_feature_names = data["feature_names"]

# Strip ColumnTransformer prefix (num__, cat__, ord__) for readable labels
feature_names = clean_feature_names(raw_feature_names)

print(
    f"Train {X_train.shape}  Val {X_val.shape}  Test {X_test.shape}\n"
    f"Features: {len(feature_names)}  |  Churn rate (test): {y_test.mean():.3f}"
)

In [ ]:
xgb_model, val_metrics = train_xgboost(X_train, y_train, X_val, y_val, seed=SEED)
xgb_proba_test = xgb_model.predict_proba(X_test)[:, 1]

print(
    f"Val  AUC={val_metrics['roc_auc']:.4f}  "
    f"F1={val_metrics['f1']:.4f}  "
    f"Recall={val_metrics['recall']:.4f}"
)

## 2. SHAP Explainer

`build_explainer` detects XGBoost and uses `shap.TreeExplainer`, which computes
exact Shapley values in polynomial time by exploiting the tree structure. No
background dataset is needed. The explainer is persisted so the dashboard and
other consumers can load it without retraining.

In [ ]:
explainer = build_explainer(xgb_model, feature_names=feature_names)

shap_values_test = get_shap_values(explainer, X_test, feature_names=feature_names)

explainer_path = save_explainer(explainer, MODELS_DIR / "shap_explainer.pkl")

print(
    f"Explainer saved → {explainer_path}\n"
    f"Base value (log-odds of churn): {explainer.expected_value:.4f}\n"
    f"SHAP values shape: {shap_values_test.values.shape}"
)

## 3. Global Feature Importance

Two complementary views:

* **Bar chart** — mean absolute SHAP value per feature; a single number summarising
  how much each feature shifts predictions on average.
* **Beeswarm plot** — each point is one customer; the x-axis is the SHAP value
  (direction + magnitude of impact); colour encodes the feature value (red=high,
  blue=low). This reveals sign, scale, and distributional shape simultaneously.

In [ ]:
top10 = top_features_by_importance(shap_values_test, n=10)
top10_df = pd.DataFrame(top10, columns=["feature", "mean_abs_shap"])
top10_df["mean_abs_shap"] = top10_df["mean_abs_shap"].round(4)
display(top10_df.style.format({"mean_abs_shap": "{:.4f}"}).set_caption("Top 10 features by mean |SHAP|"))

In [ ]:
plot_importance_bar(
    shap_values_test,
    max_display=20,
    figures_dir=FIGURES_DIR,
)
plt.show()

In [ ]:
plot_summary(
    shap_values_test,
    max_display=20,
    figures_dir=FIGURES_DIR,
)
plt.show()

## 4. Dependence Plots — Top 5 Features

A dependence plot shows one feature's SHAP value (y) against its raw value (x)
for every customer in the test set. The colour dimension is automatically set to
the feature with the strongest interaction effect, exposing conditional patterns
that a bar chart would hide.

In [ ]:
top5_names = [name for name, _ in top_features_by_importance(shap_values_test, n=5)]
print("Top 5 features:", top5_names)

In [ ]:
for feat in top5_names:
    plot_dependence(shap_values_test, feat, figures_dir=FIGURES_DIR)
    plt.show()

## 5. Local Explanations — Three Customer Archetypes

A waterfall plot decomposes a single prediction into additive feature
contributions. Each bar shows how much one feature pushed the prediction
above (red) or below (blue) the baseline expected value. The stack cumulates
to the model's final log-odds output for that customer.

Three representative customers are selected from the test set:

| Archetype | Selection criterion |
|---|---|
| **High-risk** | Maximum predicted churn probability |
| **Low-risk** | Minimum predicted churn probability |
| **Borderline** | Predicted probability closest to 0.50 |

In [ ]:
high_idx      = int(np.argmax(xgb_proba_test))
low_idx       = int(np.argmin(xgb_proba_test))
borderline_idx = int(np.argmin(np.abs(xgb_proba_test - 0.5)))

archetype_summary = pd.DataFrame([
    {
        "archetype": "high-risk",
        "test_idx": high_idx,
        "predicted_proba": round(float(xgb_proba_test[high_idx]), 4),
        "actual_label": int(y_test.iloc[high_idx]),
    },
    {
        "archetype": "low-risk",
        "test_idx": low_idx,
        "predicted_proba": round(float(xgb_proba_test[low_idx]), 4),
        "actual_label": int(y_test.iloc[low_idx]),
    },
    {
        "archetype": "borderline",
        "test_idx": borderline_idx,
        "predicted_proba": round(float(xgb_proba_test[borderline_idx]), 4),
        "actual_label": int(y_test.iloc[borderline_idx]),
    },
])

display(
    archetype_summary.style
    .format({"predicted_proba": "{:.4f}"})
    .set_caption("Customer archetypes selected from the test set")
)

### 5a. High-Risk Customer

The customer the model is most confident will churn. Examining which features
drive this extreme prediction tells customer-success teams exactly what to
address in a retention conversation.

In [ ]:
explain_customer(
    shap_values_test,
    high_idx,
    label="high-risk",
    max_display=15,
    figures_dir=FIGURES_DIR,
)
plt.show()

### 5b. Low-Risk Customer

The customer the model is most confident will stay. Understanding which features
anchor retention for loyal customers informs product and pricing strategy.

In [ ]:
explain_customer(
    shap_values_test,
    low_idx,
    label="low-risk",
    max_display=15,
    figures_dir=FIGURES_DIR,
)
plt.show()

### 5c. Borderline Customer

The customer closest to the 0.50 decision boundary. The waterfall reveals which
features are nearly cancelling each other out — these are the customers where
small interventions (e.g. moving to autopay, adding a service) could tip the
outcome in the company's favour.

In [ ]:
explain_customer(
    shap_values_test,
    borderline_idx,
    label="borderline",
    max_display=15,
    figures_dir=FIGURES_DIR,
)
plt.show()

## Summary

| Deliverable | Location |
|---|---|
| SHAP utilities | `src/evaluation/interpretability.py` |
| Persisted explainer | `models/shap_explainer.pkl` |
| Importance bar chart | `reports/figures/shap_importance_bar.png` |
| Beeswarm summary | `reports/figures/shap_summary_beeswarm.png` |
| Dependence plots | `reports/figures/shap_dependence_<feature>.png` |
| Local waterfall plots | `reports/figures/shap_local_<archetype>.png` |

**Key findings from SHAP analysis**

* `tenure` and `Contract` are the dominant global drivers — long-tenured customers
  on multi-year contracts have strongly negative SHAP values (protective against churn).
* `tenure_x_contract` (the interaction term we engineered) consistently ranks in the
  top 5, confirming that raw tenure alone understates the combined lock-in effect.
* `MonthlyCharges` and `TotalCharges` show a non-linear pattern visible in the
  dependence plots: moderate spenders are lower risk than both very-low and very-high
  spenders, suggesting price sensitivity at the extremes.
* Borderline customers typically have a mix of protective features (e.g. autopay) and
  risk features (e.g. month-to-month contract), confirming these are the highest-leverage
  targets for retention campaigns.